# LeetCode 1041 — Robot Bounded In Circle bằng Python

Link đề: https://leetcode.com/problems/robot-bounded-in-circle/ · Độ khó: **Medium** · Chủ đề: Math, String, Simulation

Skeleton LeetCode:

```python
class Solution:
    def isRobotBounded(self, instructions: str) -> bool:
        ...
```

Ghi chú: đáp án là **một giá trị `bool` duy nhất**. Robot bắt đầu ở `(0, 0)` quay mặt **hướng Bắc** và lặp đi lặp lại chuỗi lệnh `instructions` mãi mãi; ta cần biết robot có **bị giới hạn** trong một hình tròn hay không. Điểm cốt lõi: chỉ cần mô phỏng **một lượt** (hoặc bốn lượt): robot bị giới hạn ⇔ sau một lượt nó **trở lại gốc** hoặc **không còn hướng Bắc**. Oracle độc lập: mô phỏng nhiều lượt và **phát hiện chu trình** của trạng thái `(x, y, hướng)` tại cuối mỗi lượt.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ba lệnh `G` (đi thẳng 1 đơn vị), `L`, `R` (quay trái/phải 90°); lặp vô hạn chuỗi lệnh; robot có nằm trong một vòng tròn nào đó mãi mãi không.
2. Nắm công cụ: biểu diễn hướng bằng chỉ số `0..3` (Bắc, Đông, Nam, Tây) hoặc số phức, phép quay modulo `4`, tổng véc-tơ dịch chuyển, phát hiện chu trình trạng thái.
3. Đi từ **mô phỏng nhiều lượt + phát hiện chu trình** → **mô phỏng 4 lượt** → **điều kiện sau 1 lượt** → **số phức** → **hợp phép biến đổi affine**.
4. Hiểu mấu chốt: sau **một lượt**, robot ở vị trí `p` hướng `d`. Nếu `d = Bắc` thì mỗi lượt dịch thêm đúng `p`: bị giới hạn ⇔ `p = (0, 0)`. Nếu `d ≠ Bắc` thì sau tối đa `4` lượt hướng quay đủ vòng và tổng dịch chuyển **triệt tiêu** ⇒ luôn bị giới hạn.
5. Tránh lỗi kinh điển: chỉ kiểm tra vị trí về gốc (bỏ hướng), chỉ kiểm tra hướng (bỏ vị trí), mô phỏng chưa đủ `4` lượt, quên phép modulo khi quay (`d % 4`), nhầm chiều quay (L cộng, R trừ) rồi lẫn lộn với bảng hướng, đặt lại hướng mỗi lượt.

## 2. Đề bài & ví dụ

Trên mặt phẳng vô hạn, robot ban đầu ở `(0, 0)` quay mặt **hướng Bắc** (chiều dương trục `y`; Đông là chiều dương trục `x`).

Robot nhận một trong ba lệnh: `"G"` đi thẳng `1` đơn vị; `"L"` quay trái `90°` (ngược chiều kim đồng hồ); `"R"` quay phải `90°` (cùng chiều kim đồng hồ).

Robot thực hiện chuỗi `instructions` theo thứ tự và **lặp lại mãi mãi**. Trả về `true` nếu và chỉ nếu tồn tại một hình tròn trên mặt phẳng mà robot **không bao giờ rời khỏi**.

**Ví dụ 1:** `instructions = "GGLLGG"` → `true` (robot đi `(0,0) → (0,2)`, quay `180°`, về `(0,0)` quay mặt Nam: chu trình).

**Ví dụ 2:** `instructions = "GG"` → `false` (luôn đi Bắc).

**Ví dụ 3:** `instructions = "GL"` → `true`.

**Ràng buộc:** `1 <= instructions.length <= 100`; `instructions[i]` là `'G'`, `'L'` hoặc `'R'`.

In [1]:
import itertools
import random
import time
from typing import List

examples = [("GGLLGG", True), ("GG", False), ("GL", True), ("GRG", True), ("L", True), ("GLGLG", True), ("GGRGGRGGRGGR", True)]
for s, want in examples:
    print(f"instructions = {s!r} -> {want}")

instructions = 'GGLLGG' -> True
instructions = 'GG' -> False
instructions = 'GL' -> True
instructions = 'GRG' -> True
instructions = 'L' -> True
instructions = 'GLGLG' -> True
instructions = 'GGRGGRGGRGGR' -> True


## 3. Kiến thức nền

### 3.1 Hướng và phép quay

Đánh số hướng: `0 = Bắc`, `1 = Đông`, `2 = Nam`, `3 = Tây` (theo chiều kim đồng hồ). Vector bước: `DIRS = [(0, 1), (1, 0), (0, −1), (−1, 0)]`. `R` tăng chỉ số `1` (modulo `4`), `L` giảm `1` (modulo `4`, Python `-1 % 4 = 3`).

In [2]:
DIRS = [(0, 1), (1, 0), (0, -1), (-1, 0)]
NAMES = ["Bắc", "Đông", "Nam", "Tây"]


def run_once(instructions, x=0, y=0, d=0):
    """Thực hiện một lượt chuỗi lệnh từ trạng thái (x, y, d); trả về trạng thái mới."""
    for c in instructions:
        if c == "G":
            dx, dy = DIRS[d]
            x, y = x + dx, y + dy
        elif c == "L":
            d = (d - 1) % 4
        else:
            d = (d + 1) % 4
    return x, y, d


print(run_once("GGLLGG"), run_once("GG"), run_once("GL"), [NAMES[run_once(c)[2]] for c in "LRLR"])

(0, 0, 2) (0, 2, 0) (0, 1, 3) ['Tây', 'Đông', 'Tây', 'Đông']


### 3.2 Một lượt là phép biến đổi affine

Một lượt đưa trạng thái `(p, d)` tới `(p + R^d · t, d + k)` trong đó `t` là dịch chuyển của lượt tính trong hệ toạ độ ban đầu (hướng Bắc) và `k` là tổng góc quay `(mod 4)` của lượt. Do đó trạng thái sau `m` lượt được xác định bởi `t`, `k` và phép quay — ta khai thác ở mục 7 và 10.

### 3.3 Bị giới hạn ⇔ quỹ đạo hữu hạn

Robot bị giới hạn trong một hình tròn ⇔ tập vị trí robot đi qua là **hữu hạn** (bị chặn). Vì mỗi lượt xác định trạng thái `(x, y, d)` tiếp theo từ trạng thái hiện tại (`d ∈ {0,1,2,3}`), quỹ đạo của các trạng thái tại cuối mỗi lượt hoặc **lặp lại** (chu trình ⇒ bị chặn) hoặc đi ra vô hạn.

In [3]:
def orbit_states(instructions, rounds=12):
    st = (0, 0, 0)
    seen = [st]
    for _ in range(rounds):
        st = run_once(instructions, *st)
        seen.append(st)
    return seen


for s in ("GGLLGG", "GL", "GG", "GRG"):
    print(f"{s!r:10s}", orbit_states(s, 5))

'GGLLGG'   [(0, 0, 0), (0, 0, 2), (0, 0, 0), (0, 0, 2), (0, 0, 0), (0, 0, 2)]
'GL'       [(0, 0, 0), (0, 1, 3), (-1, 1, 2), (-1, 0, 1), (0, 0, 0), (0, 1, 3)]
'GG'       [(0, 0, 0), (0, 2, 0), (0, 4, 0), (0, 6, 0), (0, 8, 0), (0, 10, 0)]
'GRG'      [(0, 0, 0), (1, 1, 1), (2, 0, 2), (1, -1, 3), (0, 0, 0), (1, 1, 1)]


## 4. Ý tưởng nền tảng / trực giác

- `"GGLLGG"`: sau một lượt về `(0, 0)` hướng Nam ⇒ hướng đã đổi nên ít lượt nữa trở lại hướng Bắc và vị trí `(0,0)`: chu trình độ dài `2` ⇒ **bị giới hạn**.
- `"GG"`: sau một lượt ở `(0, 2)` hướng Bắc; mỗi lượt đi thêm `2` về phía Bắc ⇒ **không giới hạn**.
- `"GL"`: sau một lượt ở `(0, 1)` hướng Tây; bốn lượt tạo hình vuông `(0,1) → (−1,1) → (−1,0) → (0,0)` rồi lặp ⇒ giới hạn.

Quy tắc: **sau một lượt nếu robot không còn quay mặt Bắc thì quay quanh một tâm (hình vuông/hình chữ nhật); nếu vẫn quay mặt Bắc thì chỉ bị giới hạn khi chưa rời gốc**.

## 5. Brute force — mô phỏng nhiều lượt + phát hiện chu trình

**Ý tưởng:** chạy lần lượt các lượt, ghi lại trạng thái `(x, y, d)` **cuối mỗi lượt**; nếu một trạng thái lặp lại thì quỹ đạo **tuần hoàn** ⇒ bị chặn ⇒ `True`. Nếu sau `T = 200` lượt vẫn không lặp (và khoảng cách đã tăng), kết luận không giới hạn (`False`). Đây là oracle **độc lập** với định lý "4 lượt" (nó chỉ dùng định nghĩa quỹ đạo và một cận `T` rất lớn so với `4`).

In [4]:
def bounded_orbit(instructions, T=200):
    st = (0, 0, 0)
    seen = {st}
    for _ in range(T):
        st = run_once(instructions, *st)
        if st in seen:                                  # trạng thái lặp lại: quỹ đạo tuần hoàn => bị giới hạn
            return True
        seen.add(st)
    return False                                        # không lặp sau T lượt: dịch chuyển đều theo một hướng


for s, want in examples:
    assert bounded_orbit(s) == want, (s, bounded_orbit(s))
print([bounded_orbit(s) for s, _ in examples])

[True, False, True, True, True, True, True]


In [5]:
# Code nộp LeetCode cho Brute force (mô phỏng 200 lượt và phát hiện chu trình)
class SolutionOrbit:
    def isRobotBounded(self, instructions: str) -> bool:
        dirs = [(0, 1), (1, 0), (0, -1), (-1, 0)]

        def once(x, y, d):
            for c in instructions:
                if c == "G":
                    x, y = x + dirs[d][0], y + dirs[d][1]
                elif c == "L":
                    d = (d - 1) % 4
                else:
                    d = (d + 1) % 4
            return x, y, d

        st = (0, 0, 0)
        seen = {st}
        for _ in range(200):
            st = once(*st)
            if st in seen:
                return True
            seen.add(st)
        return False

**Độ phức tạp:** `O(T · n)` với `T = 200`, `n ≤ 100`, tức `2·10^4` bước; `O(T)` bộ nhớ.

**Nút thắt:** ta mô phỏng nhiều lượt trong khi chỉ cần xét **một** lượt (hoặc bốn).

## 6. Bước đột phá

Sau **một lượt** robot ở `(x, y)` hướng `d`. Nếu `d ≠ Bắc`: sau tối đa `4` lượt hướng quay đủ vòng và tổng dịch chuyển triệt tiêu ⇒ bị giới hạn. Nếu `d = Bắc`: mỗi lượt dịch thêm đúng `(x, y)` ⇒ bị giới hạn ⇔ `(x, y) = (0, 0)`. Vậy đáp án `= (x, y) == (0, 0) or d != 0`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** mô phỏng đúng `4` lượt, **Cách 3** điều kiện sau một lượt (đáp án chính), **Cách 4** số phức, **Cách 5** hợp phép biến đổi affine.

### Cách 2 — Mô phỏng đúng 4 lượt

**Giải thích:** chạy chuỗi lệnh `4` lần liên tiếp rồi kiểm tra robot có về `(0, 0)` hay không (khi đã chạy bội của `4` lượt thì hướng nếu cần cũng quay đủ vòng).

In [6]:
def bounded_four_rounds(instructions):
    x = y = d = 0
    for _ in range(4):
        x, y, d = run_once(instructions, x, y, d)
    return (x, y) == (0, 0)


for s, want in examples:
    assert bounded_four_rounds(s) == want, s
print([bounded_four_rounds(s) for s, _ in examples])

[True, False, True, True, True, True, True]


In [7]:
# Code nộp LeetCode cho Cách 2
class SolutionFourRounds:
    def isRobotBounded(self, instructions: str) -> bool:
        dirs = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        x = y = d = 0
        for _ in range(4):
            for c in instructions:
                if c == "G":
                    x, y = x + dirs[d][0], y + dirs[d][1]
                elif c == "L":
                    d = (d - 1) % 4
                else:
                    d = (d + 1) % 4
        return (x, y) == (0, 0)

**Độ phức tạp:** `O(4n)` = `O(n)` thời gian, `O(1)` bộ nhớ.

### Cách 3 — Điều kiện sau một lượt

**Ý tưởng:** mô phỏng **một** lượt, trả `(x, y) == (0, 0) or d != 0`.

In [8]:
def bounded_one_round(instructions):
    x, y, d = run_once(instructions)
    return (x, y) == (0, 0) or d != 0


for s, want in examples:
    assert bounded_one_round(s) == want, s
print([bounded_one_round(s) for s, _ in examples])

[True, False, True, True, True, True, True]


In [9]:
# Code nộp LeetCode cho Cách 3
class SolutionOneRound:
    def isRobotBounded(self, instructions: str) -> bool:
        dirs = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        x = y = d = 0
        for c in instructions:
            if c == "G":
                x, y = x + dirs[d][0], y + dirs[d][1]
            elif c == "L":
                d = (d - 1) % 4
            else:
                d = (d + 1) % 4
        return (x, y) == (0, 0) or d != 0

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### Cách 4 — Số phức

**Ý tưởng:** vị trí `z ∈ ℂ`, hướng `u` là số phức đơn vị (`1j` = Bắc); `G`: `z += u`; `L`: `u *= 1j`; `R`: `u *= −1j`. Sau một lượt: bị giới hạn ⇔ `z == 0` hoặc `u != 1j`.

In [10]:
def bounded_complex(instructions):
    z, u = 0j, 1j                                      # vị trí 0, hướng Bắc = +i
    for c in instructions:
        if c == "G":
            z += u
        elif c == "L":
            u *= 1j                                    # quay trái 90°: nhân với i
        else:
            u *= -1j                                   # quay phải 90°: nhân với -i
    return z == 0 or u != 1j


for s, want in examples:
    assert bounded_complex(s) == want, s
print([bounded_complex(s) for s, _ in examples])

[True, False, True, True, True, True, True]


In [11]:
# Code nộp LeetCode cho Cách 4
class SolutionComplex:
    def isRobotBounded(self, instructions: str) -> bool:
        z, u = 0j, 1j
        for c in instructions:
            if c == "G":
                z += u
            elif c == "L":
                u *= 1j
            else:
                u *= -1j
        return z == 0 or u != 1j

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ (số phức dùng số nguyên trong phần thực/ảo nên so sánh chính xác).

### Cách 5 — Hợp phép biến đổi affine

**Ý tưởng:** mỗi lệnh là một phép biến đổi affine `v ↦ A·v + b` trên cặp (vị trí, hướng) khi biểu diễn bằng hướng `d` và vector dịch; ta hợp thành **một** phép biến đổi `(k, t)`: `k` = tổng quay `(mod 4)`, `t` = vector dịch chuyển tổng khi bắt đầu hướng Bắc. Sau `m` lượt dịch tổng `= t + R^k t + R^{2k} t + …`; bị giới hạn ⇔ `k ≠ 0` (tổng này lặp chu kỳ và triệt tiêu sau `4/ord(k)` lượt) **hoặc** `t = 0`. Cài đặt bằng cách tính `(k, t)` rồi **tính trực tiếp** vị trí sau `4` lượt bằng công thức.

In [12]:
def rotate(vec, k):
    """Quay vector (x, y) theo chiều kim đồng hồ k lần 90°."""
    x, y = vec
    for _ in range(k % 4):
        x, y = y, -x
    return x, y


def bounded_affine(instructions):
    k, tx, ty = 0, 0, 0
    for c in instructions:
        if c == "G":
            dx, dy = rotate((0, 1), k)                 # bước đi theo hướng k
            tx, ty = tx + dx, ty + dy
        elif c == "L":
            k = (k - 1) % 4
        else:
            k = (k + 1) % 4
    # vị trí sau 4 lượt = t + R^k t + R^{2k} t + R^{3k} t
    px = py = 0
    for m in range(4):
        rx, ry = rotate((tx, ty), k * m)
        px, py = px + rx, py + ry
    return (px, py) == (0, 0)


for s, want in examples:
    assert bounded_affine(s) == want, s
print([bounded_affine(s) for s, _ in examples])

[True, False, True, True, True, True, True]


In [13]:
# Code nộp LeetCode cho Cách 5
class SolutionAffine:
    def isRobotBounded(self, instructions: str) -> bool:
        def rot(v, k):
            x, y = v
            for _ in range(k % 4):
                x, y = y, -x
            return x, y

        k, tx, ty = 0, 0, 0
        for c in instructions:
            if c == "G":
                dx, dy = rot((0, 1), k)
                tx, ty = tx + dx, ty + dy
            elif c == "L":
                k = (k - 1) % 4
            else:
                k = (k + 1) % 4
        px = py = 0
        for m in range(4):
            rx, ry = rot((tx, ty), k * m)
            px, py = px + rx, py + ry
        return (px, py) == (0, 0)

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 3** với `"GGLLGG"`:

| lệnh | `x` | `y` | hướng `d` |
|---|---|---|---|
| (đầu) | 0 | 0 | 0 (Bắc) |
| `G` | 0 | 1 | 0 |
| `G` | 0 | 2 | 0 |
| `L` | 0 | 2 | 3 (Tây) |
| `L` | 0 | 2 | 2 (Nam) |
| `G` | 0 | 1 | 2 |
| `G` | 0 | 0 | 2 |

Kết thúc ở `(0, 0)` hướng `2 ≠ 0` ⇒ `True`. Hàm trace in từng lệnh:

In [14]:
def trace_one_round(instructions):
    x = y = d = 0
    print(f"instructions = {instructions!r}: bắt đầu (0, 0) hướng {NAMES[0]}")
    for c in instructions:
        if c == "G":
            dx, dy = DIRS[d]
            x, y = x + dx, y + dy
        elif c == "L":
            d = (d - 1) % 4
        else:
            d = (d + 1) % 4
        print(f"  {c}: vị trí ({x}, {y}), hướng {NAMES[d]}")
    res = (x, y) == (0, 0) or d != 0
    print(f"=> cuối lượt: ({x}, {y}) hướng {NAMES[d]} -> {'bị giới hạn' if res else 'KHÔNG giới hạn'}")
    return res


assert trace_one_round("GGLLGG") is True
assert trace_one_round("GG") is False

instructions = 'GGLLGG': bắt đầu (0, 0) hướng Bắc
  G: vị trí (0, 1), hướng Bắc
  G: vị trí (0, 2), hướng Bắc
  L: vị trí (0, 2), hướng Tây
  L: vị trí (0, 2), hướng Nam
  G: vị trí (0, 1), hướng Nam
  G: vị trí (0, 0), hướng Nam
=> cuối lượt: (0, 0) hướng Nam -> bị giới hạn
instructions = 'GG': bắt đầu (0, 0) hướng Bắc
  G: vị trí (0, 1), hướng Bắc
  G: vị trí (0, 2), hướng Bắc
=> cuối lượt: (0, 2) hướng Bắc -> KHÔNG giới hạn


**Trace bốn lượt của `"GL"`:** lượt 1: `(0,1)` Tây; lượt 2: `(−1,1)` Nam; lượt 3: `(−1,0)` Đông; lượt 4: `(0,0)` Bắc ⇒ về gốc (đúng như `bounded_four_rounds`).

In [15]:
x = y = d = 0
for r in range(4):
    x, y, d = run_once("GL", x, y, d)
    print(f"sau lượt {r + 1}: ({x}, {y}) hướng {NAMES[d]}")
assert (x, y, d) == (0, 0, 0)

sau lượt 1: (0, 1) hướng Tây
sau lượt 2: (-1, 1) hướng Nam
sau lượt 3: (-1, 0) hướng Đông
sau lượt 4: (0, 0) hướng Bắc


## 9. Code nộp LeetCode

Chọn **Cách 3** (điều kiện sau một lượt). Giữ nguyên `class Solution`, `isRobotBounded(self, instructions)`:

In [16]:
class Solution:
    def isRobotBounded(self, instructions: str) -> bool:
        dirs = [(0, 1), (1, 0), (0, -1), (-1, 0)]         # Bắc, Đông, Nam, Tây (theo chiều kim đồng hồ)
        x = y = d = 0
        for c in instructions:                            # mô phỏng đúng MỘT lượt
            if c == "G":
                x, y = x + dirs[d][0], y + dirs[d][1]
            elif c == "L":
                d = (d - 1) % 4                           # quay trái
            else:
                d = (d + 1) % 4                           # quay phải
        # về gốc, hoặc không còn quay mặt Bắc (sau tối đa 4 lượt sẽ quay đủ vòng và về điểm xuất phát)
        return (x, y) == (0, 0) or d != 0


print([Solution().isRobotBounded(s) for s, _ in examples])
assert all(Solution().isRobotBounded(s) == w for s, w in examples)

[True, False, True, True, True, True, True]


## 10. Vì sao đúng

**(a) Mô hình.** Gọi `Φ` là hàm trạng thái sau một lượt: `(p, d) ↦ (p + R^d · t, d + k)` với `t` là dịch chuyển của lượt khi bắt đầu hướng Bắc (`d = 0`), `k` là tổng quay mod `4`, và `R` là phép quay `90°` theo chiều kim đồng hồ (quay toàn bộ véc-tơ `t` `d` lần). Lượt thứ `m + 1` bắt đầu ở trạng thái `(p_m, d_m)` với `d_m = m·k (mod 4)` và `p_{m+1} = p_m + R^{m k} t`.

**(b) Trường hợp `k = 0` (kết thúc hướng Bắc).** Khi đó `d_m = 0` mọi `m` nên `p_{m} = m·t`. Nếu `t = 0` robot luôn ở gốc đầu mỗi lượt ⇒ bị chặn (mọi vị trí giữa các lượt lấy từ một tập hữu hạn). Nếu `t ≠ 0`, `|p_m| = m|t| → ∞` ⇒ **không bị chặn**.

**(c) Trường hợp `k ≠ 0`.** Khi đó `R^k` có bậc `4 / gcd(4, k) ∈ {2, 4}` (với `k = 2`: bậc `2`; `k = 1, 3`: bậc `4`). Tổng bốn lượt liên tiếp `S = Σ_{m=0}^{3} R^{mk} t`. Với `k = 1` hoặc `3`: `{R^{mk}}` chạy qua `{I, R, R², R³}` ⇒ `S = t + R t + R² t + R³ t = 0`. Với `k = 2`: `R^{2m}` lặp `I, R², I, R²` ⇒ `S = 2(t + R² t) = 2(t − t) = 0` (vì `R² = −I`). Vậy sau mỗi `4` lượt robot trở lại đúng vị trí cũ (và hướng `d_4 = 4k = 0`): quỹ đạo là một chu trình độ dài `4`, vị trí chỉ lấy hữu hạn giá trị ⇒ **bị chặn**.

**(d) Kết luận.** Bị chặn ⇔ `k ≠ 0` hoặc `t = 0` ⇔ (hướng cuối `≠` Bắc) hoặc (vị trí cuối `= (0, 0)`) — chính là điều kiện Cách 3. Cách 2 (4 lượt): nếu `k ≠ 0` thì sau `4` lượt về gốc (theo (c)); nếu `k = 0` thì sau `4` lượt ở `4t`, về gốc ⇔ `t = 0`: đúng. Cách 4 chỉ là cách biểu diễn bằng số phức (`u ≠ i ⇔ k ≠ 0`). Cách 5 tính tổng `S` trực tiếp (theo (c) `S = 0` khi `k ≠ 0`, `S = 4t` khi `k = 0`).

**(e) Brute force.** Quỹ đạo trạng thái tại cuối mỗi lượt là dãy `(p_m, d_m)`; nếu lặp lại một trạng thái thì tuần hoàn (bị chặn). Theo (b), khi không bị chặn thì `p_m = m t` đều tăng nên không lặp trong `T = 200` lượt.

**(f) Kiểm chứng bằng máy.** Mọi cách khớp oracle quỹ đạo trên **mọi** chuỗi `{G, L, R}` độ dài `1..9` (`29 523` chuỗi) và trên chuỗi ngẫu nhiên độ dài tới `100` (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Quỹ đạo 200 lượt | `O(200·n)` | `O(200)` | oracle |
| 2. Mô phỏng 4 lượt | `O(4n)` | `O(1)` | không cần chứng minh chi tiết |
| 3. Một lượt | `O(n)` | `O(1)` | **đáp án chính** |
| 4. Số phức | `O(n)` | `O(1)` | gọn |
| 5. Hợp affine | `O(n)` | `O(1)` | cách nhìn đại số |

Đo thực tế (`n = 100`):

In [17]:
def bench(fn, arg, repeat=2000):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(arg)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(1041)
big = "".join(rng.choice("GLR") for _ in range(100))
for name, fn in [("orbit", bounded_orbit), ("four_rounds", bounded_four_rounds), ("one_round", bounded_one_round), ("complex", bounded_complex), ("affine", bounded_affine)]:
    print(f"{name:12s}: {bench(fn, big, repeat=20 if name == 'orbit' else 2000) * 1e6:9.1f} us -> {fn(big)}")

orbit       :      28.5 us -> True


four_rounds :      58.1 us -> True
one_round   :       5.7 us -> True
complex     :       5.8 us -> True
affine      :       9.6 us -> True


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi chuỗi độ dài `1..9`, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn `n = 100`.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "one_round": bounded_one_round,
    "four_rounds": bounded_four_rounds,
    "complex": bounded_complex,
    "affine": bounded_affine,
    "orbit": bounded_orbit,
    "Solution": lambda s: Solution().isRobotBounded(s),
    "SolutionOneRound": lambda s: SolutionOneRound().isRobotBounded(s),
    "SolutionFourRounds": lambda s: SolutionFourRounds().isRobotBounded(s),
    "SolutionComplex": lambda s: SolutionComplex().isRobotBounded(s),
    "SolutionAffine": lambda s: SolutionAffine().isRobotBounded(s),
    "SolutionOrbit": lambda s: SolutionOrbit().isRobotBounded(s),
}


def check(s, want=None):
    if want is None:
        want = bounded_orbit(s)
    for name in IMPLS:
        assert IMPLS[name](s) is want, (name, s, IMPLS[name](s), want)
    return want


for s, want in examples:
    check(s, want)
print("ví dụ đề: oracle quỹ đạo và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle quỹ đạo và 11 cài đặt đúng


### B. Edge case

- Chuỗi một lệnh: `"G"` (không giới hạn), `"L"`, `"R"` (giới hạn: quay tại chỗ).
- Không có `G`: robot đứng yên (bị chặn).
- `"LR"`, `"RL"`: hướng không đổi, vị trí gốc (bị chặn).
- `"GLR"`... (đi Bắc, quay trái rồi phải: hướng Bắc, vị trí `(0,1)` ⇒ không giới hạn).
- `"GRGL"`: kết thúc hướng Bắc ở `(1,1)` ⇒ không giới hạn.
- `"GLGLGLG"`: ba lần quay trái, hướng cuối khác Bắc (bị giới hạn).
- Chuỗi dài lặp `"LLLL"` (bốn lần quay trái): không đổi hướng.
- Quay `180°` giữa chuỗi (`"GGLLGG"`).

In [19]:
edge = ["G", "L", "R", "LR", "RL", "LLLL", "RRRR", "GG", "GLR", "GRGL", "GLGLGLG", "GL", "GR", "GLL", "GRR", "GGLLGG", "GGRRGG", "GLGL", "GRGRGRG", "LLGG", "GLLG", "GLRG",
        "GGLGGLGGLGGL", "GGRGGRGGRGGR", "GGGGLLGGGG", "GLGRGLGR", "G" * 100, "L" * 100, ("GLLG" * 25), "GL" * 50]
for s in edge:
    check(s)
print(len(edge), "edge case: mọi cài đặt khớp oracle quỹ đạo")
print({s: Solution().isRobotBounded(s) for s in ("G", "L", "LR", "GLR", "GRGL", "GLGLGLG")})

30 edge case: mọi cài đặt khớp oracle quỹ đạo
{'G': False, 'L': True, 'LR': True, 'GLR': False, 'GRGL': False, 'GLGLGLG': True}


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_only_origin(s):                                 # chỉ kiểm tra về gốc sau một lượt
    x, y, d = run_once(s)
    return (x, y) == (0, 0)


def bug_only_direction(s):                              # chỉ kiểm tra hướng khác Bắc
    x, y, d = run_once(s)
    return d != 0


def bug_two_rounds(s):                                  # mô phỏng chỉ 2 lượt
    x, y, d = run_once(s)
    x, y, d = run_once(s, x, y, d)
    return (x, y) == (0, 0)


def bug_three_rounds(s):                                # mô phỏng 3 lượt
    x = y = d = 0
    for _ in range(3):
        x, y, d = run_once(s, x, y, d)
    return (x, y) == (0, 0)


def bug_same_turn(s):                                   # L và R cùng quay một chiều
    x = y = d = 0
    for c in s:
        if c == "G":
            dx, dy = DIRS[d]
            x, y = x + dx, y + dy
        else:
            d = (d + 1) % 4
    return (x, y) == (0, 0) or d != 0


def bug_reset_direction(s):                             # đặt lại hướng Bắc ở mỗi lượt
    x = y = d = 0
    for _ in range(4):
        x, y, _d = run_once(s, x, y, 0)
    return (x, y) == (0, 0)


def bug_no_modulo(s):                                   # quên modulo: d có thể ra ngoài [0, 3]
    x = y = d = 0
    for c in s:
        if c == "G":
            dx, dy = DIRS[d]                            # IndexError khi d >= 4
            x, y = x + dx, y + dy
        elif c == "L":
            d -= 1
        else:
            d += 1
    return (x, y) == (0, 0) or d != 0


BUGS = [bug_only_origin, bug_only_direction, bug_two_rounds, bug_three_rounds, bug_same_turn, bug_reset_direction, bug_no_modulo]
pool = ["".join(c) for n in range(1, 8) for c in itertools.product("GLR", repeat=n)]
for bug in BUGS:
    found = None
    for s in pool:
        try:
            got = bug(s)
        except IndexError:
            got = "IndexError"
        if got != bounded_orbit(s):
            found = (s, got)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:22s} sai trên {found[0]!r}: ra {found[1]}, đúng {bounded_orbit(found[0])}")

bug_only_origin        sai trên 'GL': ra False, đúng True
bug_only_direction     sai trên 'LR': ra False, đúng True
bug_two_rounds         sai trên 'GL': ra False, đúng True
bug_three_rounds       sai trên 'GL': ra False, đúng True
bug_same_turn          sai trên 'GLR': ra True, đúng False
bug_reset_direction    sai trên 'GL': ra False, đúng True
bug_no_modulo          sai trên 'GLLLL': ra True, đúng False


### D. Vét cạn mọi chuỗi độ dài `1..9`

Mọi chuỗi trên `{G, L, R}` độ dài `1..9`: `3 + 9 + … + 3⁹ = 29 523` chuỗi. Oracle: phát hiện chu trình trạng thái cuối lượt (không dùng định lý 4 lượt). Mọi cài đặt phải khớp.

In [21]:
t0 = time.perf_counter()
cnt = true_cnt = 0
for n in range(1, 10):
    for tup in itertools.product("GLR", repeat=n):
        s = "".join(tup)
        want = bounded_orbit(s, T=40)
        for name in IMPLS:
            if name == "orbit":
                continue
            assert IMPLS[name](s) is want, (name, s, IMPLS[name](s), want)
        cnt += 1
        true_cnt += want
print(cnt, "chuỗi (độ dài 1..9):", len(IMPLS) - 1, "cài đặt khớp oracle quỹ đạo; số chuỗi bị giới hạn:", true_cnt, ";", round(time.perf_counter() - t0, 1), "s")
assert cnt == sum(3 ** n for n in range(1, 10))

29523 chuỗi (độ dài 1..9): 10 cài đặt khớp oracle quỹ đạo; số chuỗi bị giới hạn: 22468 ; 1.9 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Các cách khớp oracle quỹ đạo (`T = 200`) trên chuỗi dài tới `100`; (2) **đối xứng gương**: hoán đổi `L ↔ R` không đổi đáp án; (3) thêm `"LLLL"` hay `"RRRR"` vào bất kỳ chỗ nào không đổi đáp án; (4) **lặp chuỗi** (`s + s`, `s * 4`) không đổi đáp án; (5) chuỗi không có `G` luôn `True`; (6) chuỗi chỉ có `G` luôn `False`; (7) vị trí sau đúng `4` lượt: bằng `(0,0)` ⇔ đáp án `True`.

In [22]:
def mirror_lr(s):
    return s.translate(str.maketrans("LR", "RL"))


rng = random.Random(20240120)
t0 = time.perf_counter()
cnt_true = 0
for it in range(3000):
    n = rng.randint(1, 100)
    w = rng.choice(["GLR", "GGGLR", "GL", "GRLRL", "GGGGGGLR"])
    s = "".join(rng.choice(w) for _ in range(n))
    want = bounded_orbit(s)
    for name in IMPLS:
        assert IMPLS[name](s) is want, (name, s)                                           # (1)
    cnt_true += want
    assert Solution().isRobotBounded(mirror_lr(s)) is want                                 # (2)
    pos = rng.randint(0, n)
    assert Solution().isRobotBounded(s[:pos] + rng.choice(["LLLL", "RRRR"]) + s[pos:]) is want   # (3)
    assert Solution().isRobotBounded(s + s) is want and Solution().isRobotBounded(s * 4) is want # (4)
    x, y, d = 0, 0, 0
    for _ in range(4):
        x, y, d = run_once(s, x, y, d)
    assert ((x, y) == (0, 0)) is want                                                      # (7)
    assert Solution().isRobotBounded(s.replace("G", "")) is True if s.replace("G", "") else True    # (5)
    assert Solution().isRobotBounded("G" * n) is False                                    # (6)
print("3000 chuỗi ngẫu nhiên (độ dài tới 100): 7 tính chất đúng; số chuỗi bị giới hạn:", cnt_true, ";", round(time.perf_counter() - t0, 1), "s")

3000 chuỗi ngẫu nhiên (độ dài tới 100): 7 tính chất đúng; số chuỗi bị giới hạn: 2209 ; 2.7 s


### F. Giới hạn

Độ dài `100`: toàn `G` (không giới hạn); toàn `L`/`R`; `"GL"` lặp; chuỗi hình xoắn ốc; chuỗi có tổng dịch chuyển lớn nhưng hướng đổi (`"GGGGGGGGGGGGGGGGGGGGL"` lặp `4` lần); chuỗi "G..GLLG..G" về gốc.

In [23]:
t0 = time.perf_counter()
cases = [
    ("G" * 100, False), ("L" * 100, True), ("R" * 99, True), ("GL" * 50, True), ("G" * 50 + "LL" + "G" * 48, True), ("G" * 99 + "L", True),
    ("G" * 40 + "R" + "G" * 40 + "LR" * 9, True), ("GLGLG" * 20, True), ("GRGL" * 25, False), ("GGGLGGGR" * 12 + "GGGG", None),
]
for s, want in cases:
    got = check(s) if want is None else check(s, want)
    print(f"  n = {len(s):3d} ({s[:12]}...): {got}")
print("n ≤ 100: mọi cài đặt khớp oracle quỹ đạo và đáp án dựng sẵn;", round(time.perf_counter() - t0, 2), "s")

  n = 100 (GGGGGGGGGGGG...): False
  n = 100 (LLLLLLLLLLLL...): True
  n =  99 (RRRRRRRRRRRR...): True
  n = 100 (GLGLGLGLGLGL...): True
  n = 100 (GGGGGGGGGGGG...): True
  n = 100 (GGGGGGGGGGGG...): True
  n =  99 (GGGGGGGGGGGG...): True
  n = 100 (GLGLGGLGLGGL...): True
  n = 100 (GRGLGRGLGRGL...): False
  n = 100 (GGGLGGGRGGGL...): False
n ≤ 100: mọi cài đặt khớp oracle quỹ đạo và đáp án dựng sẵn; 0.02 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ kiểm tra về gốc

In [24]:
print("bug:", bug_only_origin("GL"), "| đúng:", Solution().isRobotBounded("GL"))
assert bug_only_origin("GL") is False and Solution().isRobotBounded("GL") is True

bug: False | đúng: True


**Vì sao sai:** `"GL"` kết thúc một lượt ở `(0, 1)` **hướng Tây**; bốn lượt tạo hình vuông và về gốc ⇒ bị giới hạn dù một lượt chưa về gốc. Phải xét cả **hướng**.

### Lỗi 2: chỉ kiểm tra hướng

In [25]:
print("bug:", bug_only_direction("LR"), "| đúng:", Solution().isRobotBounded("LR"))
assert bug_only_direction("LR") is False and Solution().isRobotBounded("LR") is True

bug: False | đúng: True


**Vì sao sai:** nếu hướng cuối vẫn là Bắc, robot chỉ bị chặn khi **không rời gốc** (như `"LR"` hay chuỗi không có `G`).

### Lỗi 3: mô phỏng chưa đủ lượt (2 hoặc 3)

In [26]:
print("2 lượt:", bug_two_rounds("GL"), "| 3 lượt:", bug_three_rounds("GL"), "| đúng:", Solution().isRobotBounded("GL"))
assert bug_two_rounds("GL") is False and bug_three_rounds("GL") is False and Solution().isRobotBounded("GL") is True

2 lượt: False | 3 lượt: False | đúng: True


**Vì sao sai:** khi mỗi lượt quay `90°`, cần **4 lượt** để hướng quay đủ vòng; 2 hoặc 3 lượt chưa về gốc. (Với quay `180°` thì 2 lượt đủ, nhưng ta phải chọn số lượt chung `4`.)

### Lỗi 4: `L` và `R` quay cùng một chiều

In [27]:
bad = next(s for s in pool if bug_same_turn(s) != bounded_orbit(s))
print("phản ví dụ:", repr(bad), "| bug:", bug_same_turn(bad), "| đúng:", bounded_orbit(bad))
assert bug_same_turn(bad) != bounded_orbit(bad)

phản ví dụ: 'GLR' | bug: True | đúng: False


**Vì sao sai:** `L` giảm chỉ số hướng, `R` tăng (hoặc ngược lại thống nhất). Cùng chiều làm sai lệch hướng cuối (ví dụ `"GLR"`).

### Lỗi 5: đặt lại hướng mỗi lượt

In [28]:
bad = next(s for s in pool if bug_reset_direction(s) != bounded_orbit(s))
print("phản ví dụ:", repr(bad), "| bug:", bug_reset_direction(bad), "| đúng:", bounded_orbit(bad))
assert bug_reset_direction(bad) != bounded_orbit(bad)

phản ví dụ: 'GL' | bug: False | đúng: True


**Vì sao sai:** robot giữ **hướng** từ cuối lượt trước sang lượt sau (mô phỏng 4 lượt phải truyền `(x, y, d)` liên tục).

### Lỗi 6 (bonus): quên modulo cho hướng

In [29]:
try:
    print("bug:", bug_no_modulo("RRRRG"))
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().isRobotBounded("RRRRG"), "(đi 1 bước về Bắc, hướng cuối Bắc, không về gốc)")
assert Solution().isRobotBounded("RRRRG") is False

IndexError: list index out of range
đúng: False (đi 1 bước về Bắc, hướng cuối Bắc, không về gốc)


**Vì sao sai:** sau nhiều lệnh `R`, chỉ số `d` có thể `≥ 4` (và `L` cho âm). Với `d = 4` truy cập `DIRS[4]` lỗi, và `d != 0` không còn nghĩa "không quay mặt Bắc" (ví dụ `d = 4` vẫn là Bắc). **Đúng:** `d = (d ± 1) % 4`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "chuỗi lệnh lặp mãi trên mặt phẳng; hỏi bị giới hạn hay không" → xét **một lượt** (hoặc bốn lượt): hướng cuối khác hướng đầu ⇒ bị giới hạn; cùng hướng ⇒ bị giới hạn ⇔ về điểm xuất phát.

**Mẹo:**

1. Hướng là chỉ số mod `4`; `L = −1`, `R = +1`; vector bước `DIRS[d]`.
2. Tổng bốn lượt khi hướng đổi = 0 (quay đủ vòng).
3. Số phức: `u *= 1j` (trái), `u *= −1j` (phải).
4. Oracle: phát hiện chu trình trạng thái `(x, y, d)` tại cuối mỗi lượt.
5. Kiểm tra bất biến: gương `L ↔ R`, thêm `"LLLL"`, lặp chuỗi.
6. Truyền `(x, y, d)` liên tục giữa các lượt (không đặt lại hướng).

**Họ bài cùng pattern:** Walking Robot Simulation (874), Robot Return to Origin (657), Spiral Matrix (54), Walking Robot Simulation II (2069), Count Collisions on a Road, Minimum Moves to Reach Target Score.

## 15. Biến thể

1. **Chu kỳ**: số lượt tối thiểu để robot quay lại trạng thái ban đầu `(0, 0, Bắc)` (`1`, `2` hoặc `4`; hoặc `∞`).
2. **Bán kính nhỏ nhất** của hình tròn tâm gốc chứa mọi vị trí robot đi qua.
3. **Vị trí sau `N` lượt** (công thức đóng) so với mô phỏng.

Cài đặt và đối chiếu mô phỏng:

In [30]:
def period(s, T=200):
    st = (0, 0, 0)
    cur = st
    for m in range(1, T + 1):
        cur = run_once(s, *cur)
        if cur == st:
            return m
    return None


def max_radius_sq(s, T=40):
    """Bình phương khoảng cách lớn nhất tới gốc trên mọi vị trí (cả giữa các lệnh) trong T lượt (đủ cho chuỗi bị giới hạn)."""
    x = y = d = 0
    best = 0
    for _ in range(T):
        for c in s:
            if c == "G":
                dx, dy = DIRS[d]
                x, y = x + dx, y + dy
                best = max(best, x * x + y * y)
            elif c == "L":
                d = (d - 1) % 4
            else:
                d = (d + 1) % 4
    return best


def position_after(s, N):
    """Vị trí sau N lượt bằng công thức đóng: t = dịch chuyển một lượt (hướng Bắc), k = quay tổng."""
    k, tx, ty = 0, 0, 0
    for c in s:
        if c == "G":
            dx, dy = rotate((0, 1), k)
            tx, ty = tx + dx, ty + dy
        elif c == "L":
            k = (k - 1) % 4
        else:
            k = (k + 1) % 4
    full, rem = divmod(N, 4)
    px = py = 0
    # mỗi khối 4 lượt cho tổng S = Σ_{m<4} R^{mk} t (bằng 0 nếu k != 0, bằng 4t nếu k = 0)
    S = (0, 0)
    for m in range(4):
        rx, ry = rotate((tx, ty), k * m)
        S = (S[0] + rx, S[1] + ry)
    px, py = full * S[0], full * S[1]
    for m in range(rem):                                  # phần còn lại: rem lượt đầu của khối kế tiếp (hướng đầu khối: full*4*k ≡ 0)
        rx, ry = rotate((tx, ty), k * m)
        px, py = px + rx, py + ry
    return px, py


rng = random.Random(15)
for _ in range(800):
    n = rng.randint(1, 12)
    s = "".join(rng.choice("GLR") for _ in range(n))
    N = rng.randint(0, 30)
    x = y = d = 0
    for _ in range(N):
        x, y, d = run_once(s, x, y, d)
    assert position_after(s, N) == (x, y), (s, N)                                       # (3)
    p = period(s)
    assert (p is not None) == bounded_orbit(s)
    if p is not None:
        assert p in (1, 2, 4)                                                           # (1) chu kỳ chỉ có thể là 1, 2 hoặc 4
    if bounded_orbit(s):
        r2 = max_radius_sq(s)
        assert r2 <= (n * 4) ** 2                                                       # (2) quỹ đạo nằm trong hình tròn bán kính <= 4n
print("chu kỳ ∈ {1, 2, 4}, bán kính hữu hạn, vị trí sau N lượt bằng công thức đóng: khớp mô phỏng")

chu kỳ ∈ {1, 2, 4}, bán kính hữu hạn, vị trí sau N lượt bằng công thức đóng: khớp mô phỏng


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quỹ đạo 200 lượt | phát hiện chu trình | `O(200n)` | `O(200)` | oracle |
| 2. Mô phỏng 4 lượt | quay đủ vòng | `O(4n)` | `O(1)` | không cần phân tích |
| 3. Một lượt | `(0,0)` hoặc hướng ≠ Bắc | `O(n)` | `O(1)` | **đáp án chính** |
| 4. Số phức | `z == 0 or u != 1j` | `O(n)` | `O(1)` | gọn |
| 5. Hợp affine | tổng 4 lượt `S` | `O(n)` | `O(1)` | cách nhìn đại số |

Mọi cách khớp oracle quỹ đạo trên `29 523` chuỗi độ dài `≤ 9` và `3 000` chuỗi ngẫu nhiên.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [31]:
print(Solution().isRobotBounded("GGLLGG"), Solution().isRobotBounded("GG"), Solution().isRobotBounded("GL"), Solution().isRobotBounded("GLR"), Solution().isRobotBounded("GLGLGGLGL"))

True False True False False


### Bài 2 (tính tay)

`instructions = "GRGL"`. (a) Mô phỏng một lượt: vị trí và hướng cuối. (b) Bị giới hạn không? (c) Mô phỏng `4` lượt để kiểm. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện mô phỏng một lượt:

In [32]:
def one_round_todo(instructions):
    dirs = [(0, 1), (1, 0), (0, -1), (-1, 0)]
    x = y = d = 0
    for c in instructions:
        pass            # TODO 1: 'G': cộng dirs[d]; 'L': d = (d - 1) % 4; 'R': d = (d + 1) % 4
    return False        # TODO 2: (x, y) == (0, 0) or d != 0


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(one_round_todo(s) == w for s, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **số phức**:

In [33]:
class MySolution:
    def isRobotBounded(self, instructions):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().isRobotBounded(s) == w for s, w in examples)

### Bài 5 (mini challenge): chu kỳ

Viết `period_of(instructions)` trả về số lượt **nhỏ nhất** `m ≥ 1` để robot quay lại trạng thái `(0, 0, Bắc)`, hoặc `None` nếu không bao giờ. Gợi ý: chu kỳ chỉ có thể là `1`, `2`, `4` (mục 15). Kiểm bằng mô phỏng.

In [34]:
def period_of(instructions):
    pass            # TODO


# Khi làm xong:
# assert period_of("GGLLGG") == 2 and period_of("GL") == 4 and period_of("GG") is None and period_of("LR") == 1

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: một lượt: hướng cuối khác Bắc ⇒ `True`.
- Mức 2: `"GLR"` kết thúc ở `(0,1)` hướng Bắc ⇒ `False`.
- Mức 3: `True False True False False` (với `"GLGLGGLGL"` kết thúc ở `(0, -1)` hướng Bắc).

**Bài 2**

- Mức 1: `G` → `(0,1)`; `R` → Đông; `G` → `(1,1)`; `L` → Bắc.
- Mức 2: kết thúc hướng Bắc ở `(1,1)` (khác gốc).
- Mức 3: không giới hạn (`False`); sau 4 lượt ở `(4, 4)`.

**Bài 3**

- Mức 1: `if c == "G": x += dirs[d][0]; y += dirs[d][1]`.
- Mức 2: `elif c == "L": d = (d - 1) % 4` và `else: d = (d + 1) % 4`.
- Mức 3: `return (x, y) == (0, 0) or d != 0`.

**Bài 4**

- Mức 1: `z, u = 0j, 1j`.
- Mức 2: `G`: `z += u`; `L`: `u *= 1j`; `R`: `u *= -1j`.
- Mức 3: `return z == 0 or u != 1j`.

**Bài 5**

- Mức 1: mô phỏng liên tiếp các lượt từ `(0, 0, 0)`.
- Mức 2: dừng khi trạng thái bằng trạng thái đầu; thử tối đa `4` lượt.
- Mức 3: không về trong `4` lượt ⇒ `None`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [35]:
vals = (Solution().isRobotBounded("GGLLGG"), Solution().isRobotBounded("GG"), Solution().isRobotBounded("GL"), Solution().isRobotBounded("GLR"), Solution().isRobotBounded("GLGLGGLGL"))
print(vals)
assert vals == (True, False, True, False, False)
for s, v in zip(("GGLLGG", "GG", "GL", "GLR", "GLGLGGLGL"), vals):
    assert bounded_orbit(s) is v
print("Bài 1 OK")

(True, False, True, False, False)
Bài 1 OK


### Bài 2

In [36]:
s = "GRGL"
x, y, d = run_once(s)
print("một lượt:", (x, y), NAMES[d])
assert (x, y, d) == (1, 1, 0) and Solution().isRobotBounded(s) is False
x = y = d = 0
for _ in range(4):
    x, y, d = run_once(s, x, y, d)
assert (x, y, d) == (4, 4, 0) and bounded_orbit(s) is False
print("Bài 2 OK: (1,1) Bắc -> không giới hạn; 4 lượt: (4,4)")

một lượt: (1, 1) Bắc
Bài 2 OK: (1,1) Bắc -> không giới hạn; 4 lượt: (4,4)


### Bài 3

In [37]:
def one_round_answer(instructions):
    dirs = [(0, 1), (1, 0), (0, -1), (-1, 0)]
    x = y = d = 0
    for c in instructions:
        if c == "G":
            x, y = x + dirs[d][0], y + dirs[d][1]
        elif c == "L":
            d = (d - 1) % 4
        else:
            d = (d + 1) % 4
    return (x, y) == (0, 0) or d != 0


assert all(one_round_answer(s) == w for s, w in examples)
for s in pool[:3000]:
    assert one_round_answer(s) == bounded_orbit(s, T=40)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def isRobotBounded(self, instructions):
        z, u = 0j, 1j
        for c in instructions:
            if c == "G":
                z += u
            elif c == "L":
                u *= 1j
            else:
                u *= -1j
        return z == 0 or u != 1j


assert all(MySolutionAnswer().isRobotBounded(s) == w for s, w in examples)
for s in pool[:3000]:
    assert MySolutionAnswer().isRobotBounded(s) == bounded_orbit(s, T=40)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [39]:
def period_of(instructions):
    st = (0, 0, 0)
    cur = st
    for m in range(1, 5):
        cur = run_once(instructions, *cur)
        if cur == st:
            return m
    return None


assert period_of("GGLLGG") == 2 and period_of("GL") == 4 and period_of("GG") is None and period_of("LR") == 1
for s in pool[:3000]:
    assert period_of(s) == period(s)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (chuỗi lệnh lặp mãi, hỏi bị giới hạn → một lượt: về gốc hoặc hướng cuối ≠ hướng đầu):**

```python
class Solution:
    def isRobotBounded(self, instructions: str) -> bool:
        dirs = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        x = y = d = 0
        for c in instructions:
            if c == "G":
                x, y = x + dirs[d][0], y + dirs[d][1]
            elif c == "L":
                d = (d - 1) % 4
            else:
                d = (d + 1) % 4
        return (x, y) == (0, 0) or d != 0
```

**Checklist trước khi submit:**

- [ ] Kiểm tra **cả** vị trí (về gốc) **và** hướng (≠ Bắc)?
- [ ] `L`: `(d − 1) % 4`, `R`: `(d + 1) % 4` (có modulo)?
- [ ] Nếu mô phỏng nhiều lượt: đủ `4` lượt và truyền `(x, y, d)` liên tục?
- [ ] Đã thử `"GGLLGG" → True`, `"GG" → False`, `"GL" → True`, `"LR"`, chuỗi không có `G`?
- [ ] Hiểu vì sao hướng đổi ⇒ chu kỳ ≤ 4 lượt (tổng quay đủ vòng triệt tiêu dịch chuyển)?
- [ ] Đúng tên `class Solution`, `isRobotBounded(self, instructions)`, không in debug.